# Notebook 2: Transform Dataset with LLM

**Objective**: Transform FinanceBench to AVI format using LLM generation.

**Outputs**:
- `data/processed/filter_rules.csv` (150 embargo policies)
- `data/processed/vector_documents.csv` (150 alternative contexts)
- `data/processed/links.csv` (150 mappings)
- `data/processed/test_queries.csv` (test dataset)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / "src"))

from transform.dataset_builder import DatasetBuilder
from utils.helpers import ensure_dir

# Load environment
load_dotenv()

print("✅ Imports ready")

## 1. Load FinanceBench

In [ ]:
input_path = Path('../data/raw/financebench_open_source.jsonl')

if not input_path.exists():
    raise FileNotFoundError(f"{input_path} not found. Run notebook 01 first.")

fb_df = pd.read_json(input_path, lines=True)
print(f"✅ Loaded {len(fb_df)} questions")

# Preview
fb_df.head()

## 2. Initialize Dataset Builder

In [ ]:
builder = DatasetBuilder()
print("✅ DatasetBuilder initialized")
print(f"   Policy Generator: {builder.policy_gen.llm.model}")
print(f"   Context Generator: {builder.context_gen.llm.model}")

## 3. Generate Rules (Embargo Policies)

⏱️ This will take ~5-10 minutes for 150 questions

In [ ]:
# Generate policies with LLM
rules_df = builder.policy_gen.generate_rules(
    fb_df,
    show_progress=True
)

print(f"\n✅ Generated {len(rules_df)} rules")
rules_df.head()

### Sample Generated Policies

In [ ]:
print("📋 Sample Generated Policies:\n")
print("=" * 80)

for i in range(3):
    print(f"\nRule {i+1}:")
    print(f"Question: {fb_df.iloc[i]['question']}")
    print(f"Policy: {rules_df.iloc[i]['text']}")
    print("-" * 80)

## 4. Generate Documents (Alternative Contexts)

⏱️ This will take another ~5-10 minutes

In [ ]:
# Generate alternative contexts with LLM
documents_df = builder.context_gen.generate_contexts(
    fb_df,
    rules_df,
    show_progress=True
)

print(f"\n✅ Generated {len(documents_df)} documents")
documents_df.head()

### Sample Generated Contexts

In [ ]:
print("📄 Sample Generated Contexts:\n")
print("=" * 80)

for i in range(3):
    print(f"\nDocument {i+1}:")
    print(f"Question: {fb_df.iloc[i]['question']}")
    print(f"Restricted Answer: {fb_df.iloc[i]['answer']}")
    print(f"Safe Context: {documents_df.iloc[i]['text']}")
    print("-" * 80)

## 5. Create Links

In [ ]:
links_df = builder.create_links(rules_df, documents_df)

print(f"✅ Created {len(links_df)} links")
links_df.head()

## 6. Save Dataset

In [ ]:
output_dir = Path('../data/processed')

builder.save_dataset(rules_df, documents_df, links_df, str(output_dir))

print(f"\n💾 Saved to {output_dir}/")
print(f"   - filter_rules.csv ({len(rules_df)} rows)")
print(f"   - vector_documents.csv ({len(documents_df)} rows)")
print(f"   - links.csv ({len(links_df)} rows)")

## 7. Create Test Queries

In [ ]:
test_queries = builder.create_test_queries(
    fb_df,
    output_path=str(output_dir / 'test_queries.csv')
)

test_queries.head()

## 8. Summary

✅ Dataset transformed to AVI format

**Created Files:**
- Rules: 150 embargo policies
- Documents: 150 alternative contexts
- Links: 150 rule→document mappings
- Test queries: 150 queries for experiment

**Next**: Upload CSVs to AVI, then run `03_run_experiment.ipynb`